# NeoOLAF × RAGTree — Unified 4-Dataset Native Benchmark v1.1

Active datasets only: **EventStoryLine, FinCausal, MAVEN-ERE, CausalBank**. DocRED is intentionally excluded from active runs.

Scientific invariants enforced by this notebook:

- no file under `src/neoolaf` is modified;
- every document uses the native NeoOLAF Layer 0–12 pipeline;
- gold `entities`, `relations`, `pred_relations`, and MAVEN `ontology_links` are stripped **before** pipeline execution;
- gold is written/read only after Layer 12 for evaluation;
- EventStoryLine reuses the already-tested v1.7 adapter;
- FinCausal uses proposition/fact endpoints and semantic `CAUSE` direction;
- MAVEN-ERE uses event/coreference-cluster endpoints and `CAUSE` / `PRECONDITION`;
- CausalBank uses no-gold lexical endpoint normalization and the full normalized dense `BECAUSE` / `THEREFORE` relation vocabulary;
- CausalBank chooses its record-level causal family only from visible `type`/text, never from gold relations;
- the single smoke-5 per dataset is protected by a persistent manifest and can resume missing records without re-running completed ones;
- **paid execution is OFF by default**.


In [1]:
from pathlib import Path
import os, sys, json, time, shutil, traceback
from pprint import pprint

def find_project_root():
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError("NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"
for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate
import ragtree_dataset_adapters_v1 as adapters
import eventstoryline_native_ablation_v1_7 as esl_v17

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Adapter self-test:")
pprint(adapters.offline_self_test())

c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
Adapter self-test:
{'cache_root': 'C:\\Users\\GALENC~1\\AppData\\Local\\Temp\\neoolaf_ragtree_unified4_v1_1',
 'causalbank_hash_a': 'EVENT_0cc175b9c0f1b6a8',
 'causalbank_relations': ['BECAUSE', 'THEREFORE'],
 'datasets': ['fincausal', 'maven_ere', 'causalbank'],
 'ok': True}


## Run controls — safe defaults

In [2]:
# PAID EXECUTION GUARD
RUN_PAID = True               # change to True only when you intentionally want API calls
RUN_MODE = "one_doc"           # one_doc | smoke5 | full
RUN_DATASETS = ["eventstoryline", "fincausal", "maven_ere", "causalbank"]

MODEL_NAME = "openai/gpt-oss-20b"
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
REASONING_EFFORT = "minimal"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 180

# For 429s: lower DOCUMENT_WORKERS first. Datasets are sequential by default.
DOCUMENT_WORKERS = 1
LAYER_WORKERS = 4
VERBOSE = True

# Manual override exists, but the manifest blocks accidental repeat smoke-5 runs.
FORCE_RUN = {
    "eventstoryline": False,
    "fincausal": False,
    "maven_ere": False,
    "causalbank": False,
}

# ESL keeps the previously selected fixed smoke documents. Other datasets freeze
# their first selected 5 records into the manifest the first time smoke5 is requested.
SMOKE_DOCUMENT_IDS = {
    "eventstoryline": [
        "EventStoryLine - 1_10ecbplus",
        "EventStoryLine - 1_11ecbplus",
        "EventStoryLine - 1_12ecbplus",
        "EventStoryLine - 1_13ecbplus",
        "EventStoryLine - 1_14ecbplus",
    ],
    "fincausal": [],
    "maven_ere": [],
    "causalbank": [],
}

assert RUN_MODE in {"one_doc", "smoke5", "full"}
assert all(k in expstate.DATASET_KEYS for k in RUN_DATASETS)
print("RUN_PAID =", RUN_PAID, "| RUN_MODE =", RUN_MODE, "| datasets =", RUN_DATASETS)

RUN_PAID = True | RUN_MODE = one_doc | datasets = ['eventstoryline', 'fincausal', 'maven_ere', 'causalbank']


## Zero-cost path + ontology + dataset preflight

In [3]:
RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)
ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)
DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)

print("RAGTREE_ROOT      :", RAGTREE_ROOT)
print("PREPROCESSED_DIR  :", PREPROCESSED_DIR)
print("ONTOLOGY_ROOT     :", ONTOLOGY_ROOT)
print("\nResolved ontologies:")
for k,v in ONTOLOGY_FILES.items(): print(f"  {k:15s} -> {v}")
print("\nResolved normalized JSONLs:")
for k,v in DATASET_FILES.items(): print(f"  {k:15s} -> {v}")

# Explicitly validate the exact ontology paths supplied for this benchmark.
expected_suffixes = {
    "eventstoryline": Path("OWLTime") / "time.ttl",
    "fincausal": Path("FIBO-CorePlus") / "fibo-core-plus.ttl",
    "maven_ere": Path("EventKG") / "EventKGSchema.ttl",
    "causalbank": Path("WordNet-Full") / "wordnet.ttl",
}
for k,suffix in expected_suffixes.items():
    assert str(ONTOLOGY_FILES[k]).replace('\\','/').endswith(str(suffix).replace('\\','/')), (k, ONTOLOGY_FILES[k])

# EventKG's original 644-byte schema uses rdfs:domain/range/subClassOf but does
# not explicitly type its class/properties. NeoOLAF's SeedOntologyLoader loads
# only explicitly typed OWL/RDFS classes and OWL object/datatype properties.
# Fail here, before any paid call, if the corrected EventKG seed is not installed.
from neoolaf.ontology.loader import SeedOntologyLoader
_eventkg_seed = SeedOntologyLoader().load(str(ONTOLOGY_FILES["maven_ere"]))
_eventkg_counts = {
    "classes": len(_eventkg_seed.classes_by_uri),
    "properties": len(_eventkg_seed.properties_by_uri),
}
print("EventKG NeoOLAF seed counts:", _eventkg_counts)
if not _eventkg_counts["classes"] and not _eventkg_counts["properties"]:
    raise RuntimeError(
        "EventKGSchema.ttl parses as RDF but NeoOLAF sees 0 explicit classes/properties. "
        "Replace it with the corrected EventKGSchema_FIXED.ttl provided with patch v1.1."
    )
print("\nOntology mapping + EventKG semantic preflight: OK")


RAGTREE_ROOT      : C:\Users\galencarmedeiro\RAGTree
PREPROCESSED_DIR  : C:\Users\galencarmedeiro\RAGTree\preprocessed
ONTOLOGY_ROOT     : C:\Users\galencarmedeiro\RAGTree\data\ontology

Resolved ontologies:
  eventstoryline  -> C:\Users\galencarmedeiro\RAGTree\data\ontology\OWLTime\time.ttl
  fincausal       -> C:\Users\galencarmedeiro\RAGTree\data\ontology\FIBO-CorePlus\fibo-core-plus.ttl
  maven_ere       -> C:\Users\galencarmedeiro\RAGTree\data\ontology\EventKG\EventKGSchema.ttl
  causalbank      -> C:\Users\galencarmedeiro\RAGTree\data\ontology\WordNet-Full\wordnet.ttl

Resolved normalized JSONLs:
  eventstoryline  -> C:\Users\galencarmedeiro\RAGTree\preprocessed\eventstoryline.jsonl
  fincausal       -> C:\Users\galencarmedeiro\RAGTree\preprocessed\fincausal.jsonl
  maven_ere       -> C:\Users\galencarmedeiro\RAGTree\preprocessed\maven_ere.jsonl
  causalbank      -> C:\Users\galencarmedeiro\RAGTree\preprocessed\causalbank.jsonl
EventKG NeoOLAF seed counts: {'classes': 1, 'propert

## Persistent budget/state guard

In [4]:
STATE_DIR = EXPERIMENT_ROOT / "state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
TEMPLATE_MANIFEST = STATE_DIR / "development_manifest_TEMPLATE_v1.json"
LIVE_MANIFEST = STATE_DIR / "development_manifest_v1.json"
manifest = expstate.load_manifest(LIVE_MANIFEST, TEMPLATE_MANIFEST)
print("Live manifest:", LIVE_MANIFEST)
pprint(manifest)

# Safety: a new paid smoke5 is not allowed until the one-doc sanity run completed.
for dataset_key in RUN_DATASETS:
    expstate.assert_run_allowed(manifest, dataset_key, RUN_MODE, force=FORCE_RUN[dataset_key])
    if RUN_MODE == "smoke5" and not manifest[dataset_key].get("one_doc_completed") and not FORCE_RUN[dataset_key]:
        raise RuntimeError(f"{dataset_key}: run one_doc successfully before the single smoke5.")
    if RUN_MODE == "one_doc" and manifest[dataset_key].get("one_doc_completed") and not FORCE_RUN[dataset_key]:
        print(f"NOTE: {dataset_key} one-doc is already completed; execution will skip it unless FORCE_RUN is True.")

Live manifest: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\state\development_manifest_v1.json
{'causalbank': {'best_version': 'unified-v1',
                'locked': False,
                'one_doc_completed': False,
                'smoke5_already_run': False,
                'smoke5_completed_record_keys': [],
                'smoke5_record_keys': [],
                'status': 'NOT_STARTED'},
 'eventstoryline': {'best_version': 'v1.7',
                    'locked': False,
                    'one_doc_completed': True,
                    'smoke5_already_run': False,
                    'smoke5_completed_record_keys': [],
                    'smoke5_record_keys': [],
                    'status': 'READY_5'},
 'fincausal': {'best_version': 'unified-v1',
               'locked': False,
               'one_doc_completed': True,
               'smoke5_already_run': False,
               'smoke5_completed_record_keys': [],
               'smoke5_record_keys': [],
            

## Offline dataset audit (gold allowed here only because no pipeline is running)

In [5]:
dataset_rows = {k: expstate.read_jsonl(v) for k,v in DATASET_FILES.items()}

def audit_dataset(k, rows):
    rel_counts = {}
    entity_counts = []
    for r in rows:
        entity_counts.append(len(r.get("entities") or {}))
        for rel,pairs in (r.get("relations") or {}).items():
            rel_counts[rel] = rel_counts.get(rel, 0) + len(pairs or [])
    return {
        "dataset": k,
        "records": len(rows),
        "relations": rel_counts,
        "mean_gold_entities": (sum(entity_counts)/len(entity_counts)) if entity_counts else 0,
        "has_ontology_links": sum("ontology_links" in r for r in rows),
    }

audit = [audit_dataset(k, dataset_rows[k]) for k in expstate.DATASET_KEYS]
for row in audit: pprint(row)

# Relation-vocabulary audit is OFFLINE only. It protects us from silently
# building an adapter against a five-document sample that omitted a full-dataset label.
expected_relation_vocab = {
    "eventstoryline": {"PRECONDITION", "FALLING_ACTION"},
    "fincausal": {"CAUSE"},
    "maven_ere": {"CAUSE", "PRECONDITION"},
    "causalbank": {"BECAUSE", "THEREFORE"},
}
for row in audit:
    observed = {str(x).upper() for x in row["relations"] if str(x).lower() not in {"null", "none", ""}}
    assert observed == expected_relation_vocab[row["dataset"]], (row["dataset"], observed, expected_relation_vocab[row["dataset"]])
print("\nFull relation-vocabulary audit: OK")

print("\nPipeline-visible sanitization check:")
for k in expstate.DATASET_KEYS:
    sample = expstate.strip_gold(dataset_rows[k][0])
    forbidden = {"entities","relations","pred_relations","ontology_links"} & set(sample)
    assert not forbidden, (k, forbidden)
    print(k, "OK; visible keys =", sorted(sample.keys()))


{'dataset': 'eventstoryline',
 'has_ontology_links': 0,
 'mean_gold_entities': 11.753950338600452,
 'records': 443,
 'relations': {'FALLING_ACTION': 4880, 'PRECONDITION': 4760, 'null': 55}}
{'dataset': 'fincausal',
 'has_ontology_links': 0,
 'mean_gold_entities': 1.9493278179937952,
 'records': 967,
 'relations': {'CAUSE': 929}}
{'dataset': 'maven_ere',
 'has_ontology_links': 3516,
 'mean_gold_entities': 23.6754835039818,
 'records': 3516,
 'relations': {'CAUSE': 8420, 'PRECONDITION': 37594}}
{'dataset': 'causalbank',
 'has_ontology_links': 0,
 'mean_gold_entities': 14.096296296296297,
 'records': 1080,
 'relations': {'BECAUSE': 43784, 'THEREFORE': 123326}}

Full relation-vocabulary audit: OK

Pipeline-visible sanitization check:
eventstoryline OK; visible keys = ['document_id', 'sentences', 'text', 'title', 'tokens', 'type']
fincausal OK; visible keys = ['document_id', 'sentences', 'text', 'title', 'tokens', 'type']
maven_ere OK; visible keys = ['document_id', 'sentences', 'text', 'ti

## Dataset configs

In [6]:
CONFIGS = {
    "eventstoryline": {
        "profile": EXPERIMENT_ROOT / "configs/eventstoryline_profile_native_ablation_v1_7.json",
        "guidance": EXPERIMENT_ROOT / "configs/guidance_eventstoryline_native_ablation_v1_7.json",
        "task": EXPERIMENT_ROOT / "configs/eventstoryline_task_guidance_v1_7.json",
        "catalog": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_aliases.json",
        "version": "v1.7",
    },
    "fincausal": {
        "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1.json",
        "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1.json",
        "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1.json",
        "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
        "version": "unified-v1",
    },
    "maven_ere": {
        "profile": EXPERIMENT_ROOT / "configs/maven_ere_profile_unified_v1.json",
        "guidance": EXPERIMENT_ROOT / "configs/maven_ere_guidance_unified_v1.json",
        "task": EXPERIMENT_ROOT / "configs/maven_ere_task_guidance_unified_v1.json",
        "catalog": EXPERIMENT_ROOT / "ontology/maven_ere_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/maven_ere_relation_aliases.json",
        "version": "unified-v1",
    },
    "causalbank": {
        "profile": EXPERIMENT_ROOT / "configs/causalbank_profile_unified_v1.json",
        "guidance": EXPERIMENT_ROOT / "configs/causalbank_guidance_unified_v1.json",
        "task": EXPERIMENT_ROOT / "configs/causalbank_task_guidance_unified_v1.json",
        "catalog": EXPERIMENT_ROOT / "ontology/causalbank_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/causalbank_relation_aliases.json",
        "version": "unified-v1.1",
    },
}
for k,cfg in CONFIGS.items():
    for name,p in cfg.items():
        if name != "version": assert Path(p).exists(), (k,name,p)
print("Config preflight: OK")

Config preflight: OK


## Runner — gold is created only after Layer 12

In [7]:
RUNS_ROOT = EXPERIMENT_ROOT / "runs" / "unified4_v1_1"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def safe_dir_name(text):
    import re
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

def run_one_record(dataset_key, gold_record):
    cfg = CONFIGS[dataset_key]
    rkey = expstate.record_key(dataset_key, gold_record)
    run_dir = RUNS_ROOT / dataset_key / RUN_MODE / safe_dir_name(rkey)
    run_dir.mkdir(parents=True, exist_ok=True)

    # CRITICAL: only sanitized no-gold record exists before the pipeline starts.
    clean_record = expstate.strip_gold(gold_record)
    input_path = run_dir / "pipeline_input_NO_GOLD.jsonl"
    expstate.write_jsonl(input_path, [clean_record])
    assert not ({"entities","relations","pred_relations","ontology_links"} & set(clean_record))
    gold_path = run_dir / "POSTHOC_GOLD_AFTER_LAYER12.jsonl"
    if gold_path.exists():
        gold_path.unlink()  # make the anti-leak boundary visible and auditable

    api_key = os.environ.get("OPENROUTER_API_KEY", "")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing. Set it in the environment before a paid run.")

    if dataset_key == "eventstoryline":
        final_state = esl_v17.run_native_pipeline(
            project_root=PROJECT_ROOT, input_jsonl=input_path, ontology_path=ONTOLOGY_FILES[dataset_key],
            profile_path=cfg["profile"], guidance_path=cfg["guidance"], task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"], relation_aliases_path=cfg["aliases"], run_dir=run_dir,
            model_name=MODEL_NAME, api_key=api_key, host=OPENROUTER_HOST, workers=LAYER_WORKERS,
            max_tokens=MAX_TOKENS, request_timeout=REQUEST_TIMEOUT, reasoning_effort=REASONING_EFFORT,
            verbose=VERBOSE, clean_run_dir=False,
        )
        # Gold appears on disk only now, after native Layer 12 returned.
        expstate.write_jsonl(gold_path, [{k:v for k,v in gold_record.items() if not k.startswith("__")}])
        summary = esl_v17.analyze_run(
            run_dir=run_dir, gold_jsonl=gold_path, catalog_path=cfg["catalog"], aliases_path=cfg["aliases"]
        )
        relation_metrics = summary.get("projected_relation_evaluation") or summary.get("strict_relation_evaluation") or {}
        endpoint_metrics = summary.get("relation_endpoint_evaluation") or summary.get("event_entity_evaluation") or {}
        result = {
            "dataset": dataset_key, "record_key": rkey, "document_id": gold_record.get("document_id"),
            "relation_metrics": relation_metrics, "endpoint_metrics": endpoint_metrics,
            "candidate_pool": summary.get("candidate_pool_coverage") or summary.get("candidate_pool") or {},
            "run_dir": str(run_dir),
        }
    else:
        final_state = adapters.run_native_pipeline_record(
            dataset_key=dataset_key, project_root=PROJECT_ROOT, input_jsonl=input_path,
            ontology_path=ONTOLOGY_FILES[dataset_key], profile_path=cfg["profile"], guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"], relation_catalog_path=cfg["catalog"], relation_aliases_path=cfg["aliases"],
            run_dir=run_dir, model_name=MODEL_NAME, api_key=api_key, host=OPENROUTER_HOST,
            workers=LAYER_WORKERS, max_tokens=MAX_TOKENS, request_timeout=REQUEST_TIMEOUT,
            reasoning_effort=REASONING_EFFORT, verbose=VERBOSE, clean_run_dir=False,
        )
        expstate.write_jsonl(gold_path, [{k:v for k,v in gold_record.items() if not k.startswith("__")}])
        result = adapters.evaluate_state(dataset_key, final_state, gold_record)
        result.update({"record_key": rkey, "document_id": gold_record.get("document_id"), "run_dir": str(run_dir)})
        adapters.write_json(run_dir / "posthoc_evaluation.json", result)
    return result

print("Runner defined. No API call has been made by this cell.")

Runner defined. No API call has been made by this cell.


## Execute selected mode

In [8]:
all_results = []

if not RUN_PAID:
    print("RUN_PAID=False -> STOPPED BEFORE ALL API CALLS. Preflight/audit only.")
else:
    for dataset_key in RUN_DATASETS:
        force = FORCE_RUN[dataset_key]
        entry = manifest[dataset_key]
        expstate.assert_run_allowed(manifest, dataset_key, RUN_MODE, force=force)

        if RUN_MODE == "one_doc" and entry.get("one_doc_completed") and not force:
            print(f"SKIP {dataset_key}: one-doc already completed (use FORCE_RUN only deliberately).")
            continue

        selected = expstate.select_records_for_mode(
            dataset_key, dataset_rows[dataset_key], manifest, RUN_MODE,
            preferred_document_ids=SMOKE_DOCUMENT_IDS.get(dataset_key) or None,
        )
        # Freeze the smoke set BEFORE paid execution. If interrupted, successful records are not re-run.
        expstate.save_manifest(LIVE_MANIFEST, manifest)

        if RUN_MODE == "smoke5":
            selected = expstate.pending_smoke_records(dataset_key, selected, manifest)
            if not selected:
                print(f"SKIP {dataset_key}: all selected smoke5 records already completed.")
                continue

        print(f"\n=== {dataset_key} | {RUN_MODE} | records to run now: {len(selected)} ===")
        for i, gold_record in enumerate(selected, 1):
            rkey = expstate.record_key(dataset_key, gold_record)
            print(f"[{i}/{len(selected)}] {rkey} | {gold_record.get('document_id')}")
            try:
                result = run_one_record(dataset_key, gold_record)
                all_results.append(result)
                expstate.mark_record_complete(manifest, dataset_key, RUN_MODE, rkey)
                expstate.save_manifest(LIVE_MANIFEST, manifest)
                print("  relation:", result.get("relation_metrics"))
                print("  endpoint:", result.get("endpoint_metrics"))
            except Exception:
                manifest[dataset_key]["status"] = "NEEDS_FIX" if RUN_MODE == "one_doc" else "SMOKE5_IN_PROGRESS"
                manifest[dataset_key]["last_error"] = traceback.format_exc()
                expstate.save_manifest(LIVE_MANIFEST, manifest)
                raise

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    summary_path = RUNS_ROOT / f"summary_{RUN_MODE}_{timestamp}.json"
    expstate.atomic_write_json(summary_path, all_results)
    print("\nSaved summary:", summary_path)
    print("Updated manifest:", LIVE_MANIFEST)

SKIP eventstoryline: one-doc already completed (use FORCE_RUN only deliberately).
SKIP fincausal: one-doc already completed (use FORCE_RUN only deliberately).

=== maven_ere | one_doc | records to run now: 1 ===
[1/1] maven_ere:0:e0ff956daea4cac8 | MAVEN_ERE - 2002c29914e6d8b5
[NeoOLAF] Run directory: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\unified4_v1_1\maven_ere\one_doc\maven_ere_0_e0ff956daea4cac8
[NeoOLAF] from_layer=0, to_layer=None, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF]

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.18s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=4; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.07s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.02s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 9 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 15 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.01s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.01s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[Neo

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.02s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=4; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.01s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\unified4_v1_1\maven_ere\one_doc\maven_ere_0_e0ff956daea4cac8\exports
[NeoOLAF] Finished layer: layer12_serialization in 0.14s
[NeoOLAF] Pipeline finished in 40.01s
[NeoOLAF] Saved checkpoint: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\unified4_v1_1\maven_ere\one_doc\maven_ere_0_e0ff956daea4cac8\checkpoints\after_selected_pipeline.pkl.gz
[NeoOLAF] Total run time: 40.11s
  relation: {'pred': 6, 'gold': 14, 'tp': 

RuntimeError: Seed ontology loaded no classes/properties: C:\Users\galencarmedeiro\RAGTree\data\ontology\WordNet-Full\wordnet.ttl

## Aggregate dashboard

In [ ]:
def compact_metric(m):
    if not isinstance(m, dict): return (None,None,None)
    return (m.get("precision"), m.get("recall"), m.get("f1"))

rows=[]
for r in all_results:
    ep=compact_metric(r.get("endpoint_metrics", {})); rel=compact_metric(r.get("relation_metrics", {}))
    rows.append({
        "dataset":r.get("dataset"), "document_id":r.get("document_id"),
        "endpoint_P":ep[0],"endpoint_R":ep[1],"endpoint_F1":ep[2],
        "relation_P":rel[0],"relation_R":rel[1],"relation_F1":rel[2],
        "run_dir":r.get("run_dir")
    })
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    pprint(rows)

print("\nManifest status:")
for k in expstate.DATASET_KEYS:
    print(k, manifest[k].get("status"), "smoke5=", manifest[k].get("smoke5_already_run"), "locked=", manifest[k].get("locked"))

## Freeze a dataset after the single smoke-5

Only do this after inspecting its one allowed smoke-5. Freezing prevents accidental development reruns and is required before `RUN_MODE = "full"`.


In [ ]:
# Example (leave commented until intentionally freezing):
# dataset_to_lock = "fincausal"
# manifest[dataset_to_lock]["locked"] = True
# manifest[dataset_to_lock]["status"] = "LOCKED"
# expstate.save_manifest(LIVE_MANIFEST, manifest)
# print(dataset_to_lock, "LOCKED")